要运行此程序，请在 L4 Google Colab Pro 实例上按“*运行时*”并按“*全部运行*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在您自己的计算机上安装 Unsloth，请按照我们的 Github 页面 [here](https://unsloth.ai/docs/get-started/install) 上的安装说明进行操作。

您将学习如何执行 [data prep](#Data)、如何执行 [train](#Train)、如何执行 [run the model](#Inference) 以及如何保存它

### 消息

隆重推出 **Unsloth Studio** - 一个新的开源、无代码 Web UI，用于训练和运行法学硕士。 [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<表><tr>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~% 2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fupload s%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia% 26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&宽度=376&dpr=3&质量=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>训练模型</b> - 无需代码</sub></td>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Z q%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26toke n%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&宽度=376&dpr=3&质量=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio 聊天 UI"></a><br><sub><b>在 Mac、Windows 和 Linux 上运行 GGUF 模型</b></sub></td>
</tr></表>

训练 MoE - DeepSeek、GLM、Qwen 和 gpt-oss 速度提高 12 倍，VRAM 减少 35%。 [Blog](https://unsloth.ai/docs/new/faster-moe)

超长上下文强化学习现已推出，上下文窗口增加了 7 倍！ [Blog](https://unsloth.ai/docs/new/grpo-long-context)

强化学习新功能：[FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

请访问我们的文档以获取所有 [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) 和 [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks)。

### 安装

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" #  [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    #  If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass #  For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
# @title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    #  If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### 不懒惰

目标：使用 OpenR1 的数学数据集，通过 GRPO 将“LFM2.5-1.2B-Instruct”转换为推理模型。

我们首先对模型进行预微调，使 GRPO 跳过尝试匹配格式 - 这会加快 GRPO 的速度。

In [ ]:
import os
os.environ['UNSLOTH_VLLM_STANDBY'] = "1" # Unsloth Standby 将 VRAM 减少 30% 以上
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # 可以增加更长的推理痕迹
lora_rank = 32 # 排名越高=越聪明，但速度越慢

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/LFM2.5-1.2B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # LoRA 16 位错误
    fast_inference = False, # 启用 vllm 快速推理
    max_lora_rank = lora_rank,
    load_in_fp8 = False, # Float8 RL / GRPO！
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # 选择任意大于 0 的数字！建议 8、16、32、64、128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "out_proj", "in_proj",
        "w1", "w2", "w3"
    ],
    lora_alpha = lora_rank*2, # *2 加快训练速度
    use_gradient_checkpointing = "unsloth", # 减少内存使用
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.8: Fast Lfm2 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.

model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Making `model.base_model.model.model` require gradients

In [ ]:
messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 4096, use_cache = True, streamer = TextStreamer(tokenizer))

<|startoftext|><|im_start|>user
Solve x^5 + 3x^4 - 10 = 3.<|im_end|>
<|im_start|>assistant
We are given the equation:

$$
x^5 + 3x^4 - 10 = 3
$$

### Step 1: Move all terms to one side

$$
x^5 + 3x^4 - 10 - 3 = 0
$$

$$
x^5 + 3x^4 - 13 = 0
$$

Now we have the equation:

$$
x^5 + 3x^4 - 13 = 0
$$

This is a fifth-degree polynomial equation, which is generally difficult to solve algebraically. Let's try to find rational roots using the Rational Root Theorem.

### Step 2: Apply Rational Root Theorem

Possible rational roots are factors of the constant term (-13) divided by factors of the leading coefficient (1):

Possible rational roots: $ \pm1, \pm13 $

Try $ x = 1 $:

$$
1^5 + 3(1)^4 - 13 = 1 + 3 - 13 = -9 \neq 0
$$

Try $ x = -1 $:

$$
(-1)^5 + 3(-1)^4 - 13 = -1 + 3 - 13 = -11 \neq 0
$$

Try $ x = 13 $: too large, not likely.

Try $ x = -13 $: also too large.

So no rational roots. Let's try to estimate the solution numerically.

### Step 

### GRPO 聊天模板
由于我们使用的是指令模型，因此我们需要要求模型在系统提示符中输出“<think>”和“</think>”标记。这将指导模型的响应并使最终答案更易于解析。

In [ ]:
reasoning_start = "<think>"
reasoning_end   = "</think>"

system_prompt = \
f"Think step by step between {reasoning_start} and {reasoning_end} and then"\
"output your answer. Output the final answer in \\boxed{} format."
system_prompt

让我们看看聊天模板在示例中的行为方式：

In [ ]:
tokenizer.apply_chat_template([
    {"role" : "system", "content" : system_prompt},
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}2"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

### 格式预微调
我们现在使用 NVIDIA 的 [Open Math Reasoning dataset](https://huggingface.co/datasets/nvidia/OpenMathReasoning) 的子集，该子集经过过滤后仅包含高质量的 DeepSeek R1 跟踪。

我们只会过滤大约 59 个左右的示例，以首先“启动”/预微调模型以理解我们的自定义 GRPO 格式。

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

# 尝试转换为数字 - 如果不是，则替换为 NaN
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors = "coerce").notnull()
# 仅选择数字
dataset = dataset.iloc[np.where(is_number)[0]]

dataset

我们必须格式化数据集以遵循 GRPO 样式格式：

In [ ]:
def format_dataset(x):
    problem = x["problem"]
    solution = x["generated_solution"]

    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : solution},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis = 1)

检查它是否有效：

In [ ]:
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)

让我们将预微调数据集截断为“max_seq_length/2”，因为我们不想要太长的推理轨迹。

请注意，这可能需要 2 分钟！

In [ ]:
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))

dataset = dataset.loc[dataset["N"] <= max_seq_length/2].copy()
dataset.shape

然后，我们对消息进行标记并将其转换为 Hugging Face 兼容的数据集格式：

In [ ]:
from datasets import Dataset

dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
dataset

现在让我们预先微调模型，使其遵循我们自定义的 GRPO 格式！

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 1, # 使用 GA 来模拟批量大小！
        warmup_steps = 10,
        # num_train_epochs = 1, # 将其设置为 1 次完整训练运行。
        max_steps = 100,
        learning_rate = 2e-4, # 长时间训练时减少至 2e-5
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # 用于 WandB 等
    ),
)

In [ ]:
trainer.train()

让我们检查模型是否已经学会遵循自定义格式：

In [ ]:
import gc
for _ in range(5):
    torch.cuda.empty_cache()
    gc.collect()

让我们使用 Unsloth 正态推理（不是 vLLM）：

In [ ]:
text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],
    tokenize = False,
    add_generation_prompt = True, # 必须添加生成
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1,
    max_new_tokens = 4096,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

是的，它确实遵循了格式！伟大的！让我们在 GRPO 步骤之前删除一些项目

In [ ]:
del dataset
torch.cuda.empty_cache()
import gc
gc.collect()

### 数据准备
<a名称=“数据”></a>

我们正在使用 Hugging Face 的 [Open R1 Math dataset](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed)。您还可以利用 OpenAI 著名的 [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train")
dataset

我们看第一行：

In [ ]:
dataset[0]["prompt"]

In [ ]:
dataset[0]["solution"]

在 GSM8K 中，我们注意到所有像 about 这样的答案都有一个 ####，所以我们提取它。但对于 Open R1 数据集，我们可以跳过下面的内容。

In [ ]:
def extract_hash_answer(text):
    # 如果“####”不在文本中：返回 None
    # return text.split("####")[1].strip()
    return text
extract_hash_answer(dataset[0]["solution"])

让我们绘制数据集！并查看第一行：

In [ ]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})
dataset[0]

我们创建一个正则表达式格式来匹配推理部分和答案：

In [ ]:
import re

# 匹配 think-close 标签后的答案，提取 \boxed{} 内容
solution_end_regex = r"</think>[\s]*" + \
    "(?:" + re.escape(tokenizer.eos_token) + ")?"

match_format = re.compile(
    r"</think>[\s]*"
    r".*?"
    r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}"
    r"[\s]*$",
    flags = re.MULTILINE | re.DOTALL
)
match_format

我们验证它是否有效：

In [ ]:
match_format.findall(
    "<think>Let me think!</think>"\
    "\boxed\{2\}",
)

In [ ]:
match_format.findall(
    "<think>Let me think!</think>"\
    "\boxed\{2\}  \n\n",
)

我们现在想要创建一个与格式完全匹配的奖励函数 - 如果成功，我们将奖励 3 分：

In [ ]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # 如果格式正确，则匹配！
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

如果失败，我们希望通过计算每个符号来奖励模型（如果它至少部分遵循格式）：

In [ ]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # 计算看到了多少个关键字 - 如果太多，我们就会受到处罚！
        # 如果我们看到 1，则加一些分！

        # 无需奖励开始标签，因为我们总是在前面添加它！
        score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        scores.append(score)
    return scores

最后，我们要提取生成的答案，并奖励或惩罚它！我们还通过比率根据答案与真实答案的接近程度来奖励它：

In [ ]:
def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        # 回答正确得5分！
        if guess == true_answer:
            score += 5.0
        # 如果看到空格则匹配，但奖励较少
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # 如果答案通过比率接近，我们也会奖励它！
            # 即，如果答案在某个范围内，则奖励它！
            try:
                ratio = float(guess) / float(true_answer)
                if   ratio >= 0.9 and ratio <= 1.1: score += 2.0
                elif ratio >= 0.8 and ratio <= 1.2: score += 1.5
                else: score -= 2.5 # 惩罚错误答案
            except:
                score -= 4.5 # 惩罚
        scores.append(score)
    return scores

另外，有时答案可能不是 1 个数字，而是像一个句子，例如“解决方案是 20 美元”-> 我们提取 20。

我们还删除了可能的逗号，例如 123,456

In [ ]:
match_numbers = re.compile(
    r".*?[\s]{0,}([-]?[\d\.\,]{1,})",
    flags = re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("\boxed{0.34}"))
print(match_numbers.findall("Answer is \(\boxed{44}\)."))
print(match_numbers.findall("The answer is -0.234"))
print(match_numbers.findall("17"))

我们现在准备我们的主函数，它将打印出生成的响应和真实答案，以及另一个奖励函数，该函数通过“float”将文本转换为浮点数，并查看它是否相同。

In [ ]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    # 每隔几步打印一次
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}"
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        # 转换为数字
        try:
            true_answer = float(true_answer.strip())
            # 删除逗号，如 123,456
            guess       = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            scores.append(0)
            continue
    return scores

获取前 90% 的提示长度，这样我们就不会意外截断它们！

即我们将删除前 10% 的长提示。

In [ ]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("最大长度 =", maximum_length)

# 仅过滤小于最大长度 90% 的样本
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

<a name="火车"></a>
### 训练模型

现在设置 GRPO Trainer 和所有配置！

In [ ]:
max_prompt_length = maximum_length + 1 # + 1 以防万一！
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1, # 增加到 4 以使训练更顺畅
    num_generations = 4, # 如果内存不足则减少
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # 设置为 1 以进行完整的训练
    max_steps = 10,
    save_steps = 100,
    report_to = "none", # 可以使用权重和偏差
    output_dir = "outputs",

    # 用于可选培训+评估
    # fp16_full_eval = 真，
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "步骤",
    # 评估步骤 = 1,
)

让我们运行训练器吧！如果向上滚动，您会看到奖励表。目标是看到“奖励”栏增加！

您可能需要等待 150 到 200 步才能执行任何操作。前 100 步您可能会获得 0 奖励。请耐心等待！

|步骤|训练损失|奖励 |奖励标准 |完成长度 |吉隆坡 |
|------|-------------|------------|------------|--------------------|----------|
| 1 | 0.000000 | 0.000000 0.125000 | 0.000000 | 0.000000 200.000000 | 0.000000 | 0.000000
| 2 | 0.000000 | 0.000000 0.072375 | 0.072375 0.248112 | 200.000000 | 0.000000 | 0.000000
| 3 | 0.000000 | 0.000000 -0.079000 | -0.079000 0.163776 | 182.500000 | 0.000005 | 0.000005

In [ ]:
# 用于可选培训+评估
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args = training_args,
    train_dataset = dataset,

    # 用于可选培训+评估
    # train_dataset = new_dataset["火车"],
    # eval_dataset = new_dataset["测试"],
)
trainer.train()

<a name="推理"></a>
### 推论
现在让我们尝试一下我们刚刚训练的模型！首先，我们先尝试一下没有任何 GRPO 训练的模型：

In [ ]:
messages = [
    {"role": "user", "content": "What is the sqrt of 101?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to(model.device)
_ = model.generate(
    **inputs,
    temperature = 0.1,
    top_k = 50,
    top_p = 0.1,
    repetition_penalty = 1.05,
    max_new_tokens = 1024,
    use_cache = True,
    streamer = TextStreamer(tokenizer)
)

现在我们刚刚用 GRPO 训练了 LoRA - 我们首先保存 LoRA！

In [ ]:
model.save_lora("grpo_saved_lora")

验证 LoRA 确实经过训练！

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # 验证 A 和 B 均非零
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

现在我们加载 LoRA 并测试：

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to(model.device)

_ = model.generate(
    **inputs,
    temperature = 0.1,
    top_k = 50,
    top_p = 0.1,
    repetition_penalty = 1.05,
    max_tokens = 1024,
    use_cache = True,
    streamer = TextStreamer(tokenizer)
)

我们的推理模型要好得多 - 它并不总是正确的，因为我们只训练了一个小时左右 - 如果我们延长序列长度并训练更长时间，那就更好了！

<a name="保存"></a>
### 保存为 float16 用于 vLLM

我们还支持直接保存到`float16`。对于 float16 选择“merged_16bit”，对于 int4 选择“merged_4bit”。我们还允许“lora”适配器作为后备。使用 `push_to_hub_merged` 上传到您的 Hugging Face 帐户！您可以前往 https://huggingface.co/settings/tokens 获取您的个人代币。有关更多部署选项，请参阅 [our docs](https://unsloth.ai/docs/basics/inference-and-deployment)。

In [ ]:
# 合并到16位
if False: model.save_pretrained_merged("lfm_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/lfm_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# 合并到4bit
if False: model.save_pretrained_merged("lfm_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/lfm_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# 只需 LoRA 适配器
if False:
    model.save_pretrained("lfm_lora")
    tokenizer.save_pretrained("lfm_lora")
if False:
    model.push_to_hub("HF_USERNAME/lfm_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/lfm_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp 转换
要保存到 `GGUF` / `llama.cpp`，我们现在原生支持它！我们克隆 `llama.cpp` 并默认将其保存到 `q8_0`。我们允许像“q4_k_m”这样的所有方法。使用`save_pretrained_gguf`进行本地保存，使用`push_to_hub_gguf`上传到HF。

一些支持的量化方法（完整列表在我们的 [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf#locally) 上）：
* `q8_0` - 快速转换。资源使用率较高，但总体上可以接受。
* `q4_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q4_K。
* `q5_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q5_K。

[**新**] 要微调并自动导出到 Ollama，请尝试我们的 [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# 保存到8位Q8_0
if False: model.save_pretrained_gguf("lfm_finetune", tokenizer,)
# 记得去 https://huggingface.co/settings/tokens 获取令牌！
# 并将 hf 更改为您的用户名！
if False: model.push_to_hub_gguf("HF_USERNAME/lfm_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# 保存到16位GGUF
if False: model.save_pretrained_gguf("lfm_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/lfm_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# 保存到 q4_k_m GGUF
if False: model.save_pretrained_gguf("lfm_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/lfm_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# 保存到多个 GGUF 选项 - 如果您想要多个，速度会更快！
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/lfm_finetune", # 将 hf 更改为您的用户名！
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

现在，使用 llama.cpp 中的“lfm_finetune.Q8_0.gguf”文件或“lfm_finetune.Q4_K_M.gguf”文件。

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1.训练自己的推理模型——Llama GRPO笔记本[Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. 将微调保存到Ollama。 [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 视觉微调 - 射线照相用例。 [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. 请参阅我们的 [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks) 上的 DPO、ORPO、持续预训练、对话微调等笔记本！

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️
</div>

  此笔记本和所有 Unsloth 笔记本均已获得 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可。